In [15]:
%pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
from openai import AzureOpenAI
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

client = AzureOpenAI(
  api_key = os.getenv("AZURE_OPENAI_API_KEY"),  
  api_version = "2023-05-15",
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
)

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = '../embedding_index_3m.json'


In [17]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [18]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [23]:
# Load the dataset
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    #Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=['text'], errors="ignore").fillna("")

In [20]:
# Fetch video
def get_videos(query: str, dataset:pd.core.frame.DataFrame, rows:int) -> pd.core.frame.DataFrame:
    video_vectors = dataset.copy()

    query_embeddings = client.embeddings.create(input=query, model = model).data[0].embedding

    #new column with calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    return video_vectors.head(rows)


In [21]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")


In [24]:
pd_vectors = load_dataset(DATASET_NAME)

while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)


Videos similar to 'What is Azure Machine Learning':
 - Azure Machine Learning Studio
   Summary: Stephanie Lindsay, a Program Manager at Azure Machine Learning, introduces the new Azure Machine Learning...
   YouTube: https://youtu.be/JNa4VV0d8T0?t=0
   Similarity: 0.8640561269200829
   Speakers: Stephanie Lindsay
 - Time Series Forecasting with Azure Machine Learning
   Summary: In this video, the speaker discusses time series forecasting with Azure Machine Learning. They demonstrate...
   YouTube: https://youtu.be/mGr_c2UnOUI?t=7
   Similarity: 0.8606104291337633
   Speakers: Hi, everyone. In this video, we are going to talk about time series forecasting with Azure Machine Learning.
 - Get Started with Azure Machine Learning with Visual Studio Code Tools
   Summary: In this episode of the AI Show, Chris, a Program Manager in Azure AI Team,...
   YouTube: https://youtu.be/u5tqeLAWLPU?t=0
   Similarity: 0.8554743568741938
   Speakers: Chris
 - What’s new with Azure Machine Learning
  